# Project 2: what Suriname sells to the world

**The question.** What does Suriname sell to the world, and how much of it comes from precious metals and stones, which include gold? Which kinds of goods made up Suriname's exports in each year from 2015 to 2025, and what share of the total came from chapter 71, precious metals and stones, which include gold?

**Who it is for.** An official at a ministry who must judge how much Suriname's export earnings depend on one kind of good. On Friday the team hands that official a one-page brief with two charts and checked numbers.

**The data.** UN Comtrade's record of Suriname's exports to the world, in US dollars, by chapter of the Harmonized System, the code that customs offices use to classify goods. The table was downloaded from the UN Comtrade public preview on 2026-11-06.

**Your team.** Four roles rotate every week: builder, data checker, AI checker and presenter. This week the data checker confirms that every sum uses the right rows.

**What you hand in on Friday.** This notebook with every CHECK passing, the brief for the ministry official with its two charts, and your AI log.

## Load the data

This cell is given. It reads the table from the course folder, or rebuilds it from UN Comtrade with one request per year if the file is not there.

Read it for shape before you run it: what goes in, what comes out, and where the work starts. Each row of the table is one chapter in one year, or the total for one year, and the column `level` says which. The `dtype` setting keeps `hs_code` as text, so that chapter `03` keeps its leading zero.

In [ ]:
import json, time, urllib.request
import pandas as pd

BASE = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
NAMES = "https://comtradeapi.un.org/files/v1/app/reference/H3.json"

def download_exports(years=range(2015, 2026)):
    """Rebuild the table from UN Comtrade: Suriname (740), exports (X), to the world (0)."""
    names = {r["id"]: r["text"] for r in json.load(urllib.request.urlopen(NAMES))["results"]}
    rows = []
    for year in years:
        url = f"{BASE}?reporterCode=740&period={year}&flowCode=X&partnerCode=0&cmdCode=TOTAL,AG2"
        for r in json.load(urllib.request.urlopen(url))["data"]:
            code = r["cmdCode"]
            rows.append({"year": year,
                         "level": "total" if code == "TOTAL" else "chapter",
                         "hs_code": code,
                         "description": "All goods" if code == "TOTAL" else names[code].split(" - ", 1)[1].strip(),
                         "export_usd": r["primaryValue"]})
        time.sleep(1.5)                      # wait between requests to the public server
    return pd.DataFrame(rows).sort_values(["year", "hs_code"]).reset_index(drop=True)

try:
    exports = pd.read_csv("https://santerre.ai/bigi-data/data/week02-suriname-exports.csv", dtype={"hs_code": str})
except FileNotFoundError:
    exports = download_exports()
exports

## TODO 1: describe the table

Write code that prints the number of rows, the column names, the first and last year, the number of rows at each level, and the number of missing values in each column. Store the number of different years in `n_years` and the number of rows with the level `total` in `n_total_rows`.

In [ ]:
# TODO 1
n_years = None
n_total_rows = None

In [ ]:
# CHECK 1
assert n_years is not None and n_total_rows is not None, "n_years or n_total_rows is still empty"
assert n_years == exports["year"].nunique()
assert n_total_rows == (exports["level"] == "total").sum()
assert n_total_rows == n_years, "each year should have exactly one total row"
print("CHECK 1 passed:", n_years, "years, one total row for each")

## TODO 2: the parts and the whole

Each year has one row with the level `total` and many rows with the level `chapter`. Store in `chapter_sum` the sum of the chapter rows for each year, and in `totals` the value of each year's total row, both indexed by year. Print them side by side and look at the difference.

In [ ]:
# TODO 2
chapter_sum = None
totals = None

In [ ]:
# CHECK 2
assert chapter_sum is not None and totals is not None, "chapter_sum or totals is still empty"
assert list(chapter_sum.index) == list(totals.index) == sorted(exports["year"].unique())
assert (chapter_sum - totals).abs().max() < 1, "the chapters should add up to the total, to within a dollar"
print("CHECK 2 passed: the chapters add up to the total in all", len(totals), "years")

## TODO 3: chart total exports

Draw a line chart of total exports by year, in millions of US dollars. The `export_usd` column is in US dollars, so divide by one million first. Give the chart a title, label both axes with their units, and name UN Comtrade as the source.

In [ ]:
# TODO 3

## TODO 4: what Suriname sells

Store in `top5` a table of the five chapters with the largest exports in 2025. It should hold each chapter's code, its description, its value and its share of the 2025 total in percent. Store their codes, largest first, in the list `top5_codes`.

Read each description in full. Some chapter names are long, and the first words do not always say what matters most.

In [ ]:
# TODO 4
top5 = None
top5_codes = None

In [ ]:
# CHECK 4
ch2025 = exports[(exports["level"] == "chapter") & (exports["year"] == 2025)]
expected = list(ch2025.nlargest(5, "export_usd")["hs_code"])
assert top5_codes is not None, "top5_codes is still empty"
assert list(top5_codes) == expected, "compare your list with the 2025 chapter rows, sorted by value"
print("CHECK 4 passed:", expected)

## TODO 5: the share of precious metals and stones

Store in `gold_share` the share of chapter 71 in each year's total exports, in percent, indexed by year. Draw it as a line chart with a title, labelled axes and the source. Then store in `low_year` and `high_year` the years with the lowest and the highest share.

Chapter 71 holds precious metals and stones, which include gold. The table cannot separate gold from the rest of the chapter.

In [ ]:
# TODO 5
gold_share = None
low_year = None
high_year = None

In [ ]:
# CHECK 5
t = exports[exports["level"] == "total"].set_index("year")["export_usd"]
c71 = exports[exports["hs_code"] == "71"].set_index("year")["export_usd"]
expected = 100 * c71 / t
assert gold_share is not None, "gold_share is still empty"
assert (gold_share.reindex(expected.index) - expected).abs().max() < 0.01
assert low_year == expected.idxmin() and high_year == expected.idxmax()
print("CHECK 5 passed: chapter 71 was", round(expected[2025], 1), "% of exports in 2025; lowest", f"{int(low_year)},", "highest", int(high_year))

## TODO 6: read the AI's code, then test it

Show your AI assistant the column names and the output of `exports.head()`, and ask it: *"Write pandas code that prints the share of chapter 71 in Suriname's total exports for each year, in percent."*

Do not run its code yet. Paste it into your AI log and write three answers next to it: which rows it adds up to get each year's total, which column it divides by which, and what unit it prints. Then run it in a new cell and compare its numbers with your answer to TODO 5. If they differ, find the line that causes the difference.

Whatever your AI wrote, measure one mistake that every reader of this table must be able to spot. A program that adds up every row of a year, the total row included, counts each dollar twice. Store in `share_if_every_row` the share of chapter 71 for each year that such a program would print.

Finally, ask the AI, without showing it the table: *"What were Suriname's three largest exports in 2025, and what share of its exports was gold?"* Write its answer in the AI log and test each part against TODO 4 and TODO 5.

In [ ]:
# TODO 6
share_if_every_row = None

In [ ]:
# CHECK 6
t = exports[exports["level"] == "total"].set_index("year")["export_usd"]
c71 = exports[exports["hs_code"] == "71"].set_index("year")["export_usd"]
expected = 100 * c71 / exports.groupby("year")["export_usd"].sum()
assert share_if_every_row is not None, "share_if_every_row is still empty"
assert (share_if_every_row.reindex(expected.index) - expected).abs().max() < 0.01
print("CHECK 6 passed: adding every row gives", round(expected[2025], 2), "% for 2025 instead of", round(100 * c71[2025] / t[2025], 1), "%")

## TODO 7: the brief for the ministry official

In a text cell below, write a brief of about 200 words for an official at a ministry who must judge how much Suriname's export earnings depend on one kind of good. Put your two charts under it. Use the numbers you computed, name the source, say what chapter 71 covers, and say one thing the data cannot tell the official.